In [2]:
import sys; sys.path.append("..")

from pathlib import Path

import pandas as pd
from IPython.display import display

from src.config import DATA_PROC, REPORTS, REL_THRESHOLD, TOP_K
from src.evaluate import evaluate_ranking, evaluate_rating
from src.models.baselines import BiasModel, Popularity, RandomRec

train = pd.read_parquet(DATA_PROC / "train.parquet")
val = pd.read_parquet(DATA_PROC / "val.parquet")
movies = pd.read_parquet(DATA_PROC / "movies.parquet")
n_users = 610
n_items = len(movies)

models = [
    ("random", RandomRec(seed=42), "seed=42"),
    ("popularity_bayes", Popularity("bayes", m=15), "mode=bayes; m=15"),
    ("popularity_count", Popularity("count", m=15), "mode=count; m=15"),
    ("bias", BiasModel(lam_i=10, lam_u=15), "lam_i=10; lam_u=15"),
]

records = []
ranking_results = {}
rating_results = {}
for model_name, model, params in models:
    model.fit(train, n_users, n_items)
    rating_scores = evaluate_rating(model, val)
    ranking_scores = evaluate_ranking(
        model, train, val, n_items=n_items, k=TOP_K, thr=REL_THRESHOLD
    )
    rating_results[model_name] = rating_scores
    ranking_results[model_name] = ranking_scores
    for metric, value in {**rating_scores, **ranking_scores}.items():
        records.append(
            {
                "model": model_name,
                "split": "val",
                "metric": metric,
                "value": value,
                "params": params,
            }
        )

results = pd.DataFrame(
    records, columns=["model", "split", "metric", "value", "params"]
)
REPORTS.mkdir(parents=True, exist_ok=True)
results.to_csv(REPORTS / "results_val.csv", index=False)

assert ranking_results["random"][f"ndcg@{TOP_K}"] < 0.01
assert ranking_results["popularity_count"][f"ndcg@{TOP_K}"] > ranking_results["random"][f"ndcg@{TOP_K}"]
assert 0.80 < rating_results["bias"]["rmse"] < 1.00

print(f"Validation rows: {len(val):,}; users: {n_users:,}; items: {n_items:,}.")
print(f"Saved {len(results)} metric rows to {REPORTS / 'results_val.csv'}.")
pivot = results.pivot(index="model", columns="metric", values="value")
display(pivot.round(4))

Validation rows: 8,304; users: 610; items: 9,742.
Saved 36 metric rows to C:\Users\KRISHAN\Desktop\Fun\Projects\CineMatch_Recommendation_System\cinematch\reports\results_val.csv.


metric,coverage,mae,map@10,ndcg@10,novelty,precision@10,recall@10,rmse,users_evaluated
model,,,,,,,,,
bias,0.0052,0.6823,0.0196,0.0371,9.1801,0.0222,0.0405,0.8876,554.0
popularity_bayes,0.0053,0.7601,0.0214,0.0404,9.0163,0.0245,0.0446,0.9829,554.0
popularity_count,0.0089,0.8187,0.0374,0.0638,8.4866,0.0327,0.0768,1.0415,554.0
random,0.4320,0.8187,0.0001,0.0004,19.8228,0.0005,0.0003,1.0415,554.0


### Findings
- **Random:** Validation RMSE is 1.0415; seeded random ranking is near zero at NDCG@10 = 0.0004.
- **Popularity (Bayesian):** Smoothing yields validation RMSE 0.9829 and NDCG@10 = 0.0404.
- **Popularity (count):** Count ranking exceeds random NDCG@10 (0.0638 vs. 0.0004), while its validation RMSE is 1.0415.
- **Bias:** Regularized user/item biases achieve validation RMSE 0.8876 and NDCG@10 = 0.0371.